# P3: PCA from Scratch with NumPy

The four template functions are implemented below, each with its written answer in place of `# YOUR ANSWER HERE`. The cells after them run the functions on the default dataset and check them against the docstring requirements.

The answers build on the Q1 result $\mathrm{Var}(u^\top X) = u^\top \Sigma u$.

In [ ]:
%matplotlib inline
import numpy as np
from matplotlib.figure import Figure

In [ ]:
# P3 Q2: Describe the Gaussian generator and observed correlation.
# Generator: samples come from a bivariate Gaussian N(mu, Sigma), by default
# mu = [2, -1] and Sigma = [[3, 1.2], [1.2, 1]]. The function draws independent
# standard normals. It then sets X = mu + L Z, where L is the
# Cholesky factor of Sigma (L L^T = Sigma). By linearity,
#     E[X] = mu  and  Cov(X) = L Cov(Z) L^T = L I L^T = Sigma,
# so X has exactly the requested distribution. Cholesky only exists for a
# symmetric positive-definite matrix, so it also checks the covariance input.
# mean and covariance are copied into new arrays, so inputs are never mutated.
#
# Observed correlation: the true correlation is
#     rho = Sigma_12 / sqrt(Sigma_11 * Sigma_22) = 1.2 / sqrt(3 * 1) = 0.693.
# With the default n = 500, seed = 0, the sample correlation is 0.701, and the
# sample covariance [[2.913, 1.173], [1.173, 0.960]] is close to the true one.
# 
def generate_data(n_samples=500, seed=0, mean=None, covariance=None):
    """Return reproducible Gaussian samples (n_samples,2) using NumPy and seed.

    Defaults: mean=[2,-1], covariance=[[3,1.2],[1.2,1]]. Custom mean has shape
    (2,), covariance (2,2) is positive definite. Do not mutate input arrays.
    """
    # Copy inputs into new float arrays so the caller's arrays are never modified.
    mean = np.array([2.0, -1.0] if mean is None else mean, dtype=float, copy=True)
    covariance = np.array([[3.0, 1.2], [1.2, 1.0]] if covariance is None else covariance,
                          dtype=float, copy=True)

    if mean.shape != (2,):
        raise ValueError(f"mean must have shape (2,), got {mean.shape}")
    if covariance.shape != (2, 2):
        raise ValueError(f"covariance must have shape (2, 2), got {covariance.shape}")
    if not np.allclose(covariance, covariance.T):
        raise ValueError("covariance must be symmetric")
    n_samples = int(n_samples)
    if n_samples < 1:
        raise ValueError("n_samples must be a positive integer")

    try:
        L = np.linalg.cholesky(covariance)
    except np.linalg.LinAlgError as err:
        raise ValueError("covariance must be positive definite") from err

    rng = np.random.default_rng(seed)
    Z = rng.standard_normal((n_samples, 2))
    return mean + Z @ L.T

In [ ]:
# P3 Q3: Link centering/covariance/eigendecomposition to the derivation and explain the plot.
# Derivation (Q1): for a unit vector u, Var(u^T X) = u^T Sigma u with
# Sigma = E[(X - mu)(X - mu)^T]. PCA picks the u that maximizes this subject to
# u^T u = 1. The Lagrangian u^T Sigma u - lambda (u^T u - 1) has gradient
# 2 Sigma u - 2 lambda u = 0, so Sigma u = lambda u: the optimal directions are
# eigenvectors of Sigma, and the variance along each one is u^T Sigma u = lambda.
#
# How each step implements it:
# 1. Centering: the derivation is written in terms of X - mu. Without it the "covariance" would include
#    mu mu^T and PC1 would point from the origin toward the mean.
# 2. Covariance: the sample version of E[(X - mu)(X - mu)^T] is
#    Sigma_hat = X~^T X~ / (n - 1), using the n - 1 divisor.
# 3. Eigendecomposition: solving Sigma_hat u = lambda u gives the stationary
#    points of the derivation. Eigenvalues are sorted in descending order,
#    so PC1 is the direction of maximum variance, and tiny negative round-off
#    values are clipped to 0, since a covariance matrix
#    is positive. Signs are fixed only for reproducibility.
# 4. Scores = centered @ components are the projections u^T (x - mu) from Q1;
#    their variances are the eigenvalues and they are uncorrelated.
def pca(X):
    """Fit NumPy PCA without mutating finite X of shape (n,2), n>=2.

    Return dict: mean (2,), centered (n,2), covariance (2,2), eigenvalues (2,),
    components (2,2), scores (n,2). Use covariance divisor n-1, descending
    eigenvalues, orthonormal eigenvectors as columns, scores=centered@components.
    Signs/tied-eigenspace bases are free. Include rank-deficient and tied cases.
    NumPy eigh/SVD are allowed; fitted library PCA is not.
    """
    X = np.array(X, dtype=float, copy=True)  
    if X.ndim != 2 or X.shape[1] != 2:
        raise ValueError(f"X must have shape (n, 2), got {X.shape}")
    n = X.shape[0]
    if n < 2:
        raise ValueError("X must have at least 2 samples")
    if not np.all(np.isfinite(X)):
        raise ValueError("X must contain only finite values")

    mean = X.mean(axis=0)
    centered = X - mean

    covariance = centered.T @ centered / (n - 1)
    covariance = (covariance + covariance.T) / 2      

    eigenvalues, components = np.linalg.eigh(covariance)

    order = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[order]
    components = components[:, order]

    eigenvalues = np.clip(eigenvalues, 0.0, None)

    pivots = np.argmax(np.abs(components), axis=0)
    signs = np.sign(components[pivots, np.arange(2)])
    signs[signs == 0] = 1.0
    components = components * signs

    scores = centered @ components
    return {
        "mean": mean,
        "centered": centered,
        "covariance": covariance,
        "eigenvalues": eigenvalues,
        "components": components,
        "scores": scores,
    }

In [ ]:
# P3 Q4: Interpret the variance fractions and dimensionality reduction tradeoff.
# The total variance is trace(Sigma) = sum of eigenvalues, because
# tr(U Lambda U^T) = tr(Lambda U^T U) = tr(Lambda) (cyclic trace property).
# Each eigenvalue is the variance along its component, so lambda_j / sum(lambda)
# is the fraction of the total variance that component j captures.
#
# Results (defaults): PC1 explains 89.4% and PC2 10.6% (true values 89.1% and
# 10.9%). The original features carry 75% and 25% of the variance, so neither
# feature alone summarizes the data; the correlation makes most of the spread
# lie along one diagonal direction, which PCA concentrates into PC1.
#

def explained_variance(eigenvalues):
    """Return same-shape fractions for nonnegative eigenvalues with positive sum."""
    values = np.asarray(eigenvalues, dtype=float)
    if not np.all(np.isfinite(values)):
        raise ValueError("eigenvalues must be finite")
    if np.any(values < 0):
        raise ValueError("eigenvalues must be nonnegative")
    total = values.sum()
    if total <= 0:
        raise ValueError("eigenvalues must have a positive sum")
    return values / total

In [ ]:
# P3 Q2/Q3/Q4: Provide labeled figures and discuss their meaning.
# Left panel (Q2/Q3): the 500 samples with the sample mean (x) and both
# principal directions drawn through it. The cloud is tilted
# upward, which is the positive correlation (r = 0.70).
#
# Right panel (Q4): bars give each component's fraction of variance (89.4% and
# 10.6%) and the black line gives the cumulative fraction (89.4% -> 100%).
# The large gap between the bars is why the 2D data can be reduced to 1D along
# PC1 with little loss. If the features were uncorrelated with equal variance,
# the bars would both be 50% and no dimension could be dropped without losing
# half the variance.

def make_plots(X, result):
    """Return a Matplotlib Figure (or sequence of Figures), without show/save.

    Plot samples and principal directions through their mean, and explained
    variance fractions. Label axes. Plot quality is manually reviewed.
    """
    X = np.asarray(X, dtype=float)
    mean = np.asarray(result["mean"], dtype=float)
    eigenvalues = np.asarray(result["eigenvalues"], dtype=float)
    components = np.asarray(result["components"], dtype=float)
    fractions = explained_variance(eigenvalues)

    colors = ["#1e8449", "#d68910"]
    fig = Figure(figsize=(13, 5.6), layout="constrained")
    ax_data, ax_var = fig.subplots(1, 2, gridspec_kw={"width_ratios": [1.35, 1]})

  
    ax_data.scatter(X[:, 0], X[:, 1], s=10, alpha=0.4, color="#3b6ea5",
                    edgecolor="none", label=f"samples (n = {len(X)})")

    spread = max(np.max(np.abs(X - mean)), 2 * np.sqrt(eigenvalues.max()))
    half = 1.1 * spread if spread > 0 else 1.0

    for j in range(2):
        u = components[:, j]
        ax_data.plot(*np.stack([mean - half * u, mean + half * u]).T, color=colors[j],
                     lw=1.2, ls="--", alpha=0.8)
        tip = mean + 2 * np.sqrt(eigenvalues[j]) * u
        ax_data.annotate("", xy=tip, xytext=mean,
                         arrowprops=dict(arrowstyle="-|>", color=colors[j], lw=2.8,
                                         mutation_scale=18))
        ax_data.plot([], [], color=colors[j], lw=2.8,
                     label=(f"PC{j + 1}: u = [{u[0]:.3f}, {u[1]:.3f}], "
                            f"λ = {eigenvalues[j]:.3f}"))

    ax_data.plot(*mean, "kx", ms=10, mew=2, label=f"mean ({mean[0]:.2f}, {mean[1]:.2f})")
    ax_data.set_xlim(mean[0] - half, mean[0] + half)
    ax_data.set_ylim(mean[1] - half, mean[1] + half)
    ax_data.set_aspect("equal")
    ax_data.set_xlabel("Feature 1 ($x_1$)")
    ax_data.set_ylabel("Feature 2 ($x_2$)")
    ax_data.set_title("Samples and principal directions through the mean\n"
                      "(arrow length = 2√λ, i.e. two standard deviations)")
    ax_data.grid(alpha=0.3)
    ax_data.legend(loc="upper left", fontsize=8.5, framealpha=0.95)

    labels = [f"PC{j + 1}" for j in range(len(fractions))]
    bars = ax_var.bar(labels, fractions, color=colors[: len(fractions)], width=0.55,
                      label="individual")
    for bar, frac in zip(bars, fractions):
        ax_var.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                    f"{frac:.1%}", ha="center", va="bottom", fontsize=11)
    ax_var.plot(labels, np.cumsum(fractions), "o-", color="black", lw=1.5,
                label="cumulative")
    ax_var.set_ylim(0, 1.3)
    ax_var.set_xlabel("Principal component")
    ax_var.set_ylabel("Fraction of total variance explained")
    ax_var.set_title("Explained variance fractions")
    ax_var.grid(alpha=0.3, axis="y")
    ax_var.set_yticks(np.linspace(0, 1, 6))
    ax_var.legend(loc="upper left", ncol=2)

    fig.suptitle("PCA of the 2D Gaussian dataset", fontsize=14)
    return fig

## Run on the default dataset

In [ ]:
X = generate_data()
result = pca(X)
fractions = explained_variance(result["eigenvalues"])

np.set_printoptions(precision=4, suppress=True)
print("X shape:            ", X.shape)
print("mean:               ", result["mean"])
print("covariance:\n", result["covariance"])
print("sample correlation: ", round(np.corrcoef(X, rowvar=False)[0, 1], 4),
      "  (true:", round(1.2 / np.sqrt(3), 4), ")")
print("eigenvalues:        ", result["eigenvalues"])
print("components (columns):\n", result["components"])
print("explained variance: ", fractions)

In [ ]:
fig = make_plots(X, result)
fig